After manipulating the original data in an older version of the code, it was discovered that rolling sums were not computed correctly, as they mistakenly included the data from the current game. This is not true to how we would be realistically predicting games and it sways the model, so this notebook corrects that error.

If you are using this notebook past version 1.0.0, you do not need to worry about running this notebook, as your data should be fine.

In [1]:
import pandas as pd
import numpy as np

import os
import glob

pd.set_option('display.max_columns', 1000)
pd.set_option('display.max_rows', 1000)

In [2]:
directory = '/users/blaizelahman/Desktop/CFB Model/Original Data'
pattern = os.path.join(directory, '*model*.csv')

teamFiles = glob.glob(pattern)

teamDict = {}

for file in teamFiles:

    teamDF = pd.read_csv(file)

    if teamDF.shape[0] >= 56:

        # throwing out first 20 rows due to missing rolling_sum data
        teamDF = teamDF.drop(teamDF.index[:20])
        teamDF = teamDF.reset_index(drop = True)
    
        key = teamDF['School'][0]
        teamDict[key] = teamDF
    
        print(f'Added: {key}')

Added: Boston College
Added: Rutgers
Added: Auburn
Added: North Texas
Added: Nebraska
Added: Oklahoma State
Added: Arizona State
Added: Eastern Michigan
Added: Louisiana
Added: Colorado State
Added: Idaho
Added: Illinois
Added: Air Force
Added: Kent State
Added: Louisiana Monroe
Added: Iowa
Added: Akron
Added: Ohio
Added: Georgia
Added: South Alabama
Added: Georgia Tech
Added: Western Kentucky
Added: Maryland
Added: Arizona
Added: Minnesota
Added: Pittsburgh
Added: Marshall
Added: Louisiana Tech
Added: Virginia Tech
Added: California
Added: Georgia Southern
Added: Rice
Added: Missouri
Added: UCF
Added: Kansas State
Added: Appalachian State
Added: Michigan
Added: Clemson
Added: Oregon
Added: Florida Atlantic
Added: Central Michigan
Added: Louisville
Added: Navy
Added: Washington State
Added: Tennessee
Added: Arkansas State
Added: Kansas
Added: Miami
Added: Alabama
Added: Vanderbilt
Added: USC
Added: Indiana
Added: UMass
Added: Utah State
Added: Baylor
Added: Tulsa
Added: Ole Miss
Added:

Let's see where we went wrong earlier

In [10]:
teamDict['Florida State']['Points'][-10:]

198    31
199    39
200    41
201    38
202    41
203    24
204    27
205    58
206    24
207    16
Name: Points, dtype: int64

In [11]:
teamDict['Florida State']['rolling_sum_Points8'][-10:]

198    350.0
199    344.0
200    347.0
201    336.0
202    332.0
203    311.0
204    272.0
205    299.0
206    292.0
207    269.0
Name: rolling_sum_Points8, dtype: float64

With how rolling sum is supposed to work, the rolling sum assigned to a game should reflect the sum of a given stat in the previous x games, however we mistakenly included the previous x-1 games plus the current game in that calculation. Let's fix that.

This funciton has been updated to fix the error

In [13]:
def customRollingSum(series, window):
    
    result = []
    
    for i in range(len(series)):
        
        if i < window - 1:
            result.append(np.nan)
            
        else:
            result.append(series[i-window:i].sum(skipna=True))
            
    return pd.Series(result, index=series.index)

In [15]:
for team, teamDF in teamDict.items():

    
    # making rolling sum columns for selected stats for the past 20 and 8 games  
    for games in [20, 8]:
        for column in ['Points','firstDowns','fumblesLost','fumblesRecovered','interceptions','kickReturnYards','kickingPoints','netPassingYards',
                      'passesDeflected', 'passesIntercepted','passingTDs','puntReturns','qbHurries','rushingAttempts','rushingTDs','rushingYards',
                       'sacks','tacklesForLoss','totalFumbles','totalPenaltiesYards','totalYards','turnovers','yardsPerPass','yardsPerRushAttempt', 'totalTDs']:
            newColumn = 'rolling_sum_' + column + str(games) 
            teamDF[newColumn] = customRollingSum(teamDF[column], games)
            
    # altering yardsPerPass and yardsPerRushAttempt columns to reflect their values over the past 20 and 8 games
    teamDF['rolling_sum_yardsPerPass20'] = teamDF['rolling_sum_yardsPerPass20'] / 20
    teamDF['rolling_sum_yardsPerPass8'] = teamDF['rolling_sum_yardsPerPass8'] / 8

    teamDF['rolling_sum_yardsPerRushAttempt20'] = teamDF['rolling_sum_yardsPerRushAttempt20'] / 20
    teamDF['rolling_sum_yardsPerRushAttempt8'] = teamDF['rolling_sum_yardsPerRushAttempt8'] / 8

    teamDict[team] = teamDF
    print(team)



Boston College
Rutgers
Auburn
North Texas
Nebraska
Oklahoma State
Arizona State
Eastern Michigan
Louisiana
Colorado State
Idaho
Illinois
Air Force
Kent State
Louisiana Monroe
Iowa
Akron
Ohio
Georgia
South Alabama
Georgia Tech
Western Kentucky
Maryland
Arizona
Minnesota
Pittsburgh
Marshall
Louisiana Tech
Virginia Tech
California
Georgia Southern
Rice
Missouri
UCF
Kansas State
Appalachian State
Michigan
Clemson
Oregon
Florida Atlantic
Central Michigan
Louisville
Navy
Washington State
Tennessee
Arkansas State
Kansas
Miami
Alabama
Vanderbilt
USC
Indiana
UMass
Utah State
Baylor
Tulsa
Ole Miss
BYU
UTEP
Syracuse
Middle Tennessee
Arkansas
UT San Antonio
Hawai'i
Oregon State
Southern Mississippi
Florida State
Texas A&M
UAB
Boise State
Duke
Colorado
Northern Illinois
Coastal Carolina
Florida
Wyoming
Houston
Cincinnati
New Mexico
Texas Tech
Texas State
Temple
Wisconsin
Purdue
Liberty
East Carolina
Old Dominion
Oklahoma
Toledo
Troy
Ohio State
Penn State
Buffalo
Nevada
Florida International
Iowa St

In [16]:
teamDict['Florida State']['Points'][-10:]

198    31
199    39
200    41
201    38
202    41
203    24
204    27
205    58
206    24
207    16
Name: Points, dtype: int64

In [17]:
teamDict['Florida State']['rolling_sum_Points8'][-10:]

198    360.0
199    350.0
200    344.0
201    347.0
202    336.0
203    332.0
204    311.0
205    272.0
206    299.0
207    292.0
Name: rolling_sum_Points8, dtype: float64

Need to fix the "two weeks 1's" issue

In [25]:
teamDict['Florida State'][teamDict['Florida State']['Year'] == 2022][:2]

,Unnamed: 0,Game Id,School,Conference,HomeAway,Points,Week,Year,completionAttempts,defensiveTDs,firstDowns,fourthDownEff,fumblesLost,fumblesRecovered,interceptionTDs,interceptionYards,interceptions,kickReturnTDs,kickReturnYards,kickReturns,kickingPoints,netPassingYards,passesDeflected,passesIntercepted,passingTDs,possessionTime,puntReturnTDs,puntReturnYards,puntReturns,qbHurries,rushingAttempts,rushingTDs,rushingYards,sacks,tackles,tacklesForLoss,thirdDownEff,totalFumbles,totalPenaltiesYards,totalYards,turnovers,yardsPerPass,yardsPerRushAttempt,totalTDs,School_opp,Conference_opp,HomeAway_opp,Points_opp,Week_opp,Year_opp,completionAttempts_opp,defensiveTDs_opp,firstDowns_opp,fourthDownEff_opp,fumblesLost_opp,fumblesRecovered_opp,interceptionTDs_opp,interceptionYards_opp,interceptions_opp,kickReturnTDs_opp,kickReturnYards_opp,kickReturns_opp,kickingPoints_opp,netPassingYards_opp,passesDeflected_opp,passesIntercepted_opp,passingTDs_opp,possessionTime_opp,puntReturnTDs_opp,puntReturnYards_opp,puntReturns_opp,qbHurries_opp,rushingAttempts_opp,rushingTDs_opp,rushingYards_opp,sacks_opp,tackles_opp,tacklesForLoss_opp,thirdDownEff_opp,totalFumbles_opp,totalPenaltiesYards_opp,totalYards_opp,turnovers_opp,yardsPerPass_opp,yardsPerRushAttempt_opp,totalTDs_opp,scoreDiff,pointTotal,Win,rolling_sum_Points20,rolling_sum_firstDowns20,rolling_sum_fumblesLost20,rolling_sum_fumblesRecovered20,rolling_sum_interceptions20,rolling_sum_kickReturnYards20,rolling_sum_kickingPoints20,rolling_sum_netPassingYards20,rolling_sum_passesDeflected20,rolling_sum_passesIntercepted20,rolling_sum_passingTDs20,rolling_sum_puntReturns20,rolling_sum_qbHurries20,rolling_sum_rushingAttempts20,rolling_sum_rushingTDs20,rolling_sum_rushingYards20,rolling_sum_sacks20,rolling_sum_tacklesForLoss20,rolling_sum_totalFumbles20,rolling_sum_totalPenaltiesYards20,rolling_sum_totalYards20,rolling_sum_turnovers20,rolling_sum_yardsPerPass20,rolling_sum_yardsPerRushAttempt20,rolling_sum_totalTDs20,rolling_sum_Points8,rolling_sum_firstDowns8,rolling_sum_fumblesLost8,rolling_sum_fumblesRecovered8,rolling_sum_interceptions8,rolling_sum_kickReturnYards8,rolling_sum_kickingPoints8,rolling_sum_netPassingYards8,rolling_sum_passesDeflected8,rolling_sum_passesIntercepted8,rolling_sum_passingTDs8,rolling_sum_puntReturns8,rolling_sum_qbHurries8,rolling_sum_rushingAttempts8,rolling_sum_rushingTDs8,rolling_sum_rushingYards8,rolling_sum_sacks8,rolling_sum_tacklesForLoss8,rolling_sum_totalFumbles8,rolling_sum_totalPenaltiesYards8,rolling_sum_totalYards8,rolling_sum_turnovers8,rolling_sum_yardsPerPass8,rolling_sum_yardsPerRushAttempt8,rolling_sum_totalTDs8,rolling_sum_Points20_opp,rolling_sum_firstDowns20_opp,rolling_sum_fumblesLost20_opp,rolling_sum_fumblesRecovered20_opp,rolling_sum_interceptions20_opp,rolling_sum_kickReturnYards20_opp,rolling_sum_kickingPoints20_opp,rolling_sum_netPassingYards20_opp,rolling_sum_passesDeflected20_opp,rolling_sum_passesIntercepted20_opp,rolling_sum_passingTDs20_opp,rolling_sum_puntReturns20_opp,rolling_sum_qbHurries20_opp,rolling_sum_rushingAttempts20_opp,rolling_sum_rushingTDs20_opp,rolling_sum_rushingYards20_opp,rolling_sum_sacks20_opp,rolling_sum_tacklesForLoss20_opp,rolling_sum_totalFumbles20_opp,rolling_sum_totalPenaltiesYards20_opp,rolling_sum_totalYards20_opp,rolling_sum_turnovers20_opp,rolling_sum_yardsPerPass20_opp,rolling_sum_yardsPerRushAttempt20_opp,rolling_sum_totalTDs20_opp,rolling_sum_Points8_opp,rolling_sum_firstDowns8_opp,rolling_sum_fumblesLost8_opp,rolling_sum_fumblesRecovered8_opp,rolling_sum_interceptions8_opp,rolling_sum_kickReturnYards8_opp,rolling_sum_kickingPoints8_opp,rolling_sum_netPassingYards8_opp,rolling_sum_passesDeflected8_opp,rolling_sum_passesIntercepted8_opp,rolling_sum_passingTDs8_opp,rolling_sum_puntReturns8_opp,rolling_sum_qbHurries8_opp,rolling_sum_rushingAttempts8_opp,rolling_sum_rushingTDs8_opp,rolling_sum_rushingYards8_opp,rolling_sum_sacks8_opp,rolling_sum_tacklesForLoss8_opp,rolling_sum_totalFumbles8_opp,rolling_sum_totalP

Can see that our data source labeled week 0 games as the "second week 1 game". Let's correctly label it as a week 0 game.

In [42]:
for team, teamDF in teamDict.items():
    for year in np.unique(teamDF['Year'].values):

        yearDF = teamDF[teamDF['Year'] == year]

        if np.sum(yearDF['Week'].values == 1) > 1:

            teamWeek0 = yearDF.index[0]
            teamWeek1 = yearDF.index[1]

            teamDF.loc[teamWeek0], teamDF.loc[teamWeek1] = teamDF.loc[teamWeek1].copy(), teamDF.loc[teamWeek0].copy()

            teamDF.at[teamWeek0, 'Week'] = 0
            
            print(f'Added week 0 for {team} in year {year}')

Added week 0 for North Texas in year 2022
Added week 0 for Nebraska in year 2021
Added week 0 for Nebraska in year 2022
Added week 0 for Colorado State in year 2017
Added week 0 for Colorado State in year 2018
Added week 0 for Illinois in year 2021
Added week 0 for Illinois in year 2022
Added week 0 for Ohio in year 2023
Added week 0 for Western Kentucky in year 2022
Added week 0 for Louisiana Tech in year 2023
Added week 0 for Rice in year 2018
Added week 0 for Florida Atlantic in year 2022
Added week 0 for Vanderbilt in year 2022
Added week 0 for Vanderbilt in year 2023
Added week 0 for USC in year 2023
Added week 0 for UMass in year 2017
Added week 0 for UMass in year 2018
Added week 0 for UMass in year 2023
Added week 0 for Utah State in year 2022
Added week 0 for BYU in year 2017
Added week 0 for UTEP in year 2021
Added week 0 for UTEP in year 2022
Added week 0 for UTEP in year 2023
Added week 0 for Hawai'i in year 2016
Added week 0 for Hawai'i in year 2017
Added week 0 for Hawai'

In [43]:
teamDict['Florida State'][teamDict['Florida State']['Year'] == 2022][:2]

,Unnamed: 0,Game Id,School,Conference,HomeAway,Points,Week,Year,completionAttempts,defensiveTDs,firstDowns,fourthDownEff,fumblesLost,fumblesRecovered,interceptionTDs,interceptionYards,interceptions,kickReturnTDs,kickReturnYards,kickReturns,kickingPoints,netPassingYards,passesDeflected,passesIntercepted,passingTDs,possessionTime,puntReturnTDs,puntReturnYards,puntReturns,qbHurries,rushingAttempts,rushingTDs,rushingYards,sacks,tackles,tacklesForLoss,thirdDownEff,totalFumbles,totalPenaltiesYards,totalYards,turnovers,yardsPerPass,yardsPerRushAttempt,totalTDs,School_opp,Conference_opp,HomeAway_opp,Points_opp,Week_opp,Year_opp,completionAttempts_opp,defensiveTDs_opp,firstDowns_opp,fourthDownEff_opp,fumblesLost_opp,fumblesRecovered_opp,interceptionTDs_opp,interceptionYards_opp,interceptions_opp,kickReturnTDs_opp,kickReturnYards_opp,kickReturns_opp,kickingPoints_opp,netPassingYards_opp,passesDeflected_opp,passesIntercepted_opp,passingTDs_opp,possessionTime_opp,puntReturnTDs_opp,puntReturnYards_opp,puntReturns_opp,qbHurries_opp,rushingAttempts_opp,rushingTDs_opp,rushingYards_opp,sacks_opp,tackles_opp,tacklesForLoss_opp,thirdDownEff_opp,totalFumbles_opp,totalPenaltiesYards_opp,totalYards_opp,turnovers_opp,yardsPerPass_opp,yardsPerRushAttempt_opp,totalTDs_opp,scoreDiff,pointTotal,Win,rolling_sum_Points20,rolling_sum_firstDowns20,rolling_sum_fumblesLost20,rolling_sum_fumblesRecovered20,rolling_sum_interceptions20,rolling_sum_kickReturnYards20,rolling_sum_kickingPoints20,rolling_sum_netPassingYards20,rolling_sum_passesDeflected20,rolling_sum_passesIntercepted20,rolling_sum_passingTDs20,rolling_sum_puntReturns20,rolling_sum_qbHurries20,rolling_sum_rushingAttempts20,rolling_sum_rushingTDs20,rolling_sum_rushingYards20,rolling_sum_sacks20,rolling_sum_tacklesForLoss20,rolling_sum_totalFumbles20,rolling_sum_totalPenaltiesYards20,rolling_sum_totalYards20,rolling_sum_turnovers20,rolling_sum_yardsPerPass20,rolling_sum_yardsPerRushAttempt20,rolling_sum_totalTDs20,rolling_sum_Points8,rolling_sum_firstDowns8,rolling_sum_fumblesLost8,rolling_sum_fumblesRecovered8,rolling_sum_interceptions8,rolling_sum_kickReturnYards8,rolling_sum_kickingPoints8,rolling_sum_netPassingYards8,rolling_sum_passesDeflected8,rolling_sum_passesIntercepted8,rolling_sum_passingTDs8,rolling_sum_puntReturns8,rolling_sum_qbHurries8,rolling_sum_rushingAttempts8,rolling_sum_rushingTDs8,rolling_sum_rushingYards8,rolling_sum_sacks8,rolling_sum_tacklesForLoss8,rolling_sum_totalFumbles8,rolling_sum_totalPenaltiesYards8,rolling_sum_totalYards8,rolling_sum_turnovers8,rolling_sum_yardsPerPass8,rolling_sum_yardsPerRushAttempt8,rolling_sum_totalTDs8,rolling_sum_Points20_opp,rolling_sum_firstDowns20_opp,rolling_sum_fumblesLost20_opp,rolling_sum_fumblesRecovered20_opp,rolling_sum_interceptions20_opp,rolling_sum_kickReturnYards20_opp,rolling_sum_kickingPoints20_opp,rolling_sum_netPassingYards20_opp,rolling_sum_passesDeflected20_opp,rolling_sum_passesIntercepted20_opp,rolling_sum_passingTDs20_opp,rolling_sum_puntReturns20_opp,rolling_sum_qbHurries20_opp,rolling_sum_rushingAttempts20_opp,rolling_sum_rushingTDs20_opp,rolling_sum_rushingYards20_opp,rolling_sum_sacks20_opp,rolling_sum_tacklesForLoss20_opp,rolling_sum_totalFumbles20_opp,rolling_sum_totalPenaltiesYards20_opp,rolling_sum_totalYards20_opp,rolling_sum_turnovers20_opp,rolling_sum_yardsPerPass20_opp,rolling_sum_yardsPerRushAttempt20_opp,rolling_sum_totalTDs20_opp,rolling_sum_Points8_opp,rolling_sum_firstDowns8_opp,rolling_sum_fumblesLost8_opp,rolling_sum_fumblesRecovered8_opp,rolling_sum_interceptions8_opp,rolling_sum_kickReturnYards8_opp,rolling_sum_kickingPoints8_opp,rolling_sum_netPassingYards8_opp,rolling_sum_passesDeflected8_opp,rolling_sum_passesIntercepted8_opp,rolling_sum_passingTDs8_opp,rolling_sum_puntReturns8_opp,rolling_sum_qbHurries8_opp,rolling_sum_rushingAttempts8_opp,rolling_sum_rushingTDs8_opp,rolling_sum_rushingYards8_opp,rolling_sum_sacks8_opp,rolling_sum_tacklesForLoss8_opp,rolling_sum_totalFumbles8_opp,rolling_sum_totalP

We need to add betting data to the data frames now, also from collegefootballdata.com

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait

In [ ]:
# setting the download directory and Chrome settings
directory = "/Users/blaizelahman/Desktop/CFBData"
chromeOptions = webdriver.ChromeOptions()
prefs = {"download.default_directory": directory}
chromeOptions.add_experimental_option("prefs", prefs)

# creating Chrome driver
driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()), options = chromeOptions)

# making a dictionary to hold team talent rankings
bettingDict = {}

# downloading betting data for years 2035-2023 from collegefootballdata.com
for year in range(2013, 2024):

    # skipping 2020 because it has bad data
    if year == 2020: 
        continue
        
    try:
        url = f'https://collegefootballdata.com/exporter/lines?year={year}&seasonType=regular}'
        driver.get(url)
        time.sleep(4) 
            
        # clicking the query button
        query = driver.find_element(By.XPATH, "//button[contains(span/text(), 'Query')]")
        query.click()
        time.sleep(3) 
            
        # clicking the export button
        export = driver.find_element(By.XPATH, "//button[contains(span/text(), 'Export')]")
        export.click()
        time.sleep(3)

        key = str(year)
            
        # grabs files from CFBData folder
        files = os.listdir(directory)
        
        # grab the file paths for all files ending in .csv
        filePaths = [os.path.join(directory, name) for name in files if name.endswith('.csv')]

        # grabbing the most recently made file out of those in paths
        file = max(filePaths, key=os.path.getctime)
            
        # loading csv file
        bettingDict[key] = pd.read_csv(file)

        # deleting the file after it has been added
        os.remove(file)

    except Exception as e:
        print(f'Cannot grab data for {year}. Error: {e}')

    print(f'Successfully grabbed data for {year}')

driver.quit()

We can now see that rolling sums have been calculated as intended and the week 0 issue has been fixed. Let's replace our old faulty data files with these.

In [44]:
files = glob.glob(os.path.join('/users/blaizelahman/Desktop/CFB Model/Original Data', '*'))

for file in files:
        
    try:
            
        os.remove(file)
        print(f"Deleted {file}")
            
    except Exception as e:
        
        print(f"Could not delete {file}: {e}")

Deleted /users/blaizelahman/Desktop/CFB Model/Original Data/Boston_College_model.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Original Data/Rutgers_model.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Original Data/Auburn_model.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Original Data/North_Texas_model.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Original Data/Nebraska_model.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Original Data/Oklahoma_State_model.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Original Data/Arizona_State_model.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Original Data/Eastern_Michigan_model.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Original Data/Louisiana_model.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Original Data/Colorado_State_model.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Original Data/Idaho_model.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Original Data/Illinois_model.csv
Deleted /users/blaizela

In [46]:
for key, team in teamDict.items():
    if 2023 in team['Year'].values:
        name = key.replace(' ', '_') + '_model.csv'
        path = os.path.join('/users/blaizelahman/Desktop/CFB Model/Original Data', name)
        team.to_csv(path)
        print('CSV: ' + name)

CSV: Boston_College_model.csv
CSV: Rutgers_model.csv
CSV: Auburn_model.csv
CSV: North_Texas_model.csv
CSV: Nebraska_model.csv
CSV: Oklahoma_State_model.csv
CSV: Arizona_State_model.csv
CSV: Eastern_Michigan_model.csv
CSV: Louisiana_model.csv
CSV: Colorado_State_model.csv
CSV: Idaho_model.csv
CSV: Illinois_model.csv
CSV: Air_Force_model.csv
CSV: Kent_State_model.csv
CSV: Louisiana_Monroe_model.csv
CSV: Iowa_model.csv
CSV: Akron_model.csv
CSV: Ohio_model.csv
CSV: Georgia_model.csv
CSV: South_Alabama_model.csv
CSV: Georgia_Tech_model.csv
CSV: Western_Kentucky_model.csv
CSV: Maryland_model.csv
CSV: Arizona_model.csv
CSV: Minnesota_model.csv
CSV: Pittsburgh_model.csv
CSV: Marshall_model.csv
CSV: Louisiana_Tech_model.csv
CSV: Virginia_Tech_model.csv
CSV: California_model.csv
CSV: Georgia_Southern_model.csv
CSV: Rice_model.csv
CSV: Missouri_model.csv
CSV: UCF_model.csv
CSV: Kansas_State_model.csv
CSV: Appalachian_State_model.csv
CSV: Michigan_model.csv
CSV: Clemson_model.csv
CSV: Oregon_model